# Feature engineering

This notebook captures the feature engineering steps from the exploration work, while intentionally preserving all rows across all normalized use categories. The original exploration subset to residential-only rows, but this notebook keeps the full dataset so the modeling pipeline is not restricted by `use_category_normalized`.


In [6]:
## 1) Load source data and keep full use-category coverage
import numpy as np
import pandas as pd

als = pd.read_csv("../data/ALS_Concat_Cleaned.csv", header=0)
landsale = pd.read_csv("../data/landsale.csv", header=0, na_values="N.A.")
cleaned_lot_numbers = pd.read_csv("../data/lot_mappings.csv", header=0)

# Read CPI file (World Bank format with metadata rows and years as columns)
# This is a price index used as the deflator for inflation adjustment
hk_cpi_wide = pd.read_csv("../data/hk_cpi_index.csv", skiprows=4)
# Reshape from wide (years as columns) to long format
year_cols = [str(year) for year in range(1960, 2026)]
hk_cpi = hk_cpi_wide[['Country Name'] + year_cols].melt(
    id_vars=['Country Name'],
    var_name='Year',
    value_name='Deflator'
)
hk_cpi['Year'] = hk_cpi['Year'].astype(int)
hk_cpi = hk_cpi[['Year', 'Deflator']].drop_duplicates()

# Read FEDFUNDS data
fed = pd.read_csv("../data/FEDFUNDS.csv")

# Read RGDP file and filter to Hong Kong data
rgdp_wide = pd.read_csv("../data/Real_GDP.csv", skiprows=4)
hk_rgdp = rgdp_wide[rgdp_wide['Country Name'] == 'Hong Kong SAR, China'][['Country Name'] + year_cols]
rgdp = hk_rgdp.melt(
    id_vars=['Country Name'],
    var_name='Year',
    value_name='RGDP(Billions)'
)
rgdp['Year'] = rgdp['Year'].astype(int)
rgdp = rgdp[['Year', 'RGDP(Billions)']].drop_duplicates()

# Do not subset on use_category_normalized here; keep the full sample for modeling.
print("ALS rows:", len(als))
print("Landsale rows:", len(landsale))
print("Unique normalized lot mappings:", len(cleaned_lot_numbers))
print("Deflator data points:", len(hk_cpi))
print("RGDP data points:", len(rgdp))

ALS rows: 541
Landsale rows: 1007
Unique normalized lot mappings: 1107
Deflator data points: 66
RGDP data points: 66


In [7]:
## 2) Prepare the landsale table and normalize the lot key
cols = [
    'premium_hkd_m', 'num_bids', 'successful_tenderer', 'buyer_normalized', 'buyer_type',
    'use_category_normalized', 'latitude', 'longitude', 'parent_developer', 'award_date',
    'sale_regime', 'site_area_sqm', 'lot_number', 'financial_year', 'location',
    'max_gfa', 'accommodation_value'
]

ls = landsale[cols].copy()
ls['award_date'] = pd.to_datetime(ls['award_date'], format='mixed')
ls = (
    ls
    .merge(cleaned_lot_numbers, left_on='lot_number', right_on='Original', how='left')
    .drop(columns=['Original', 'Source'])
    .rename(columns={'Clean': 'lot_number_normalized'})
)
ls['lot_number_normalized'].isna().sum()


np.int64(0)

In [8]:
## 3) Prepare the ALS table and create the ALS lot-key match
als = als.drop(columns=["Unnamed: 0"], errors='ignore')
als = (
    als
    .merge(cleaned_lot_numbers, left_on='Lot Number', right_on='Original', how='left')
    .drop(columns=['Original', 'Source'])
    .rename(columns={'Clean': 'lot_number_normalized'})
)
als['als_site_area_sqm'] = als['Site Area'].str.replace(' ha', '', regex=False).astype(float) * 10000
als['Estimated Earliest Site Available Date'] = als['Estimated Earliest Site Available Date'].replace('Now', np.nan)
als['Estimated Earliest Site Available Date'] = als.groupby('Lot Number')['Estimated Earliest Site Available Date'].transform('first')
als['Estimated Earliest Site Available Date'] = pd.to_datetime(als['Estimated Earliest Site Available Date'], format='mixed').dt.strftime('%b-%Y')

als2 = als.drop_duplicates(subset=['lot_number_normalized'], keep='first').copy()
ls['in_als'] = np.where(ls['lot_number_normalized'].isin(als2['lot_number_normalized']), 1, 0)

# Full merged table retained; no use-category filter applied here.
total = ls.merge(als2, on='lot_number_normalized', how='left')

merge_results = pd.DataFrame({
    'Total Matches': total['in_als'].sum(),
    'ALS Unmatch Count': als2[~als2['lot_number_normalized'].isin(set(ls['lot_number_normalized']))].shape[0],
    'Land Sale Unmatched Count': total[total['in_als'] == 0].shape[0],
    'Total Unique ALS Lot Numbers': als2.shape[0],
    'Total Land Sale Records in Period': ls.shape[0],
}, index=[0]).T.reset_index()
merge_results.columns = ['Index', 'Count']
merge_results


,Index,Count
0,Total Matches,152
1,ALS Unmatch Count,37
2,Land Sale Unmatched Count,855
3,Total Unique ALS Lot Numbers,189
4,Total Land Sale Records in Period,1007


In [9]:
## 4) Create full dataset with Year column
full_df = total.copy()
full_df['Year'] = full_df['award_date'].dt.strftime('%Y').astype(int)
full_df = full_df.sort_values(by='award_date', ascending=True)

full_df[['award_date', 'Year', 'lot_number_normalized', 'in_als']].head()


,award_date,Year,lot_number_normalized,in_als
0,1985-04-18,1985,STTL275,0
1,1985-04-18,1985,IL8571,0
2,1985-04-18,1985,STTL194,0
3,1985-04-18,1985,STTL225,0
4,1985-04-18,1985,STTL252,0


In [10]:
## 5) Add inflation-adjusted price features
hk_cpi['Deflator'] = hk_cpi['Deflator'].astype(float)

full_df = full_df.merge(hk_cpi[['Year', 'Deflator']], on='Year', how='left')
full_df['adj_premium_hkd_m'] = full_df['premium_hkd_m'] / full_df['Deflator']
full_df['adj_premium_per_sqm'] = full_df['adj_premium_hkd_m'] * 1_000_000 / full_df['site_area_sqm']

full_df[['award_date', 'premium_hkd_m', 'Deflator', 'adj_premium_hkd_m', 'site_area_sqm', 'adj_premium_per_sqm']].head()


,award_date,premium_hkd_m,Deflator,adj_premium_hkd_m,site_area_sqm,adj_premium_per_sqm
0,1985-04-18,12.10,39.752975,0.304380,4915.0,61.928735
1,1985-04-18,701.28,39.752975,17.640944,10690.0,1650.228609
2,1985-04-18,45.50,39.752975,1.144568,11710.0,97.742820
3,1985-04-18,27.00,39.752975,0.679194,9515.0,71.381445
4,1985-04-18,1.80,39.752975,0.045280,960.0,47.166281


In [11]:
## 6) Add macroeconomic timing variables
fed['observation_date'] = pd.to_datetime(fed['observation_date'])
full_df = full_df.sort_values(by='award_date', ascending=True)
full_df = pd.merge_asof(
    left=full_df,
    right=fed[['observation_date', 'FEDFUNDS']],
    left_on='award_date',
    right_on='observation_date',
    direction='backward'
).drop(columns=['observation_date'])

rgdp['Year'] = rgdp['Year'].astype(int)
full_df = full_df.merge(rgdp, on='Year', how='left')
full_df['RGDP(Billions)'] = full_df['RGDP(Billions)'] / 1_000_000_000

full_df[['award_date', 'Year', 'FEDFUNDS', 'RGDP(Billions)']].head()


,award_date,Year,FEDFUNDS,RGDP(Billions)
0,1985-04-18,1985,8.27,83.952508
1,1985-04-18,1985,8.27,83.952508
2,1985-04-18,1985,8.27,83.952508
3,1985-04-18,1985,8.27,83.952508
4,1985-04-18,1985,8.27,83.952508


In [12]:
## 7) Add market competition features from bid counts
# The exploration notebook created this at the financial-year level using awarded deals.
awarded = full_df.copy()
awarded = awarded[awarded['num_bids'].notna()].copy()

# Ensure num_bids is numeric
awarded['num_bids'] = pd.to_numeric(awarded['num_bids'], errors='coerce')

bids_by_fy = (
    awarded.assign(num_bids=awarded['num_bids'])
    .groupby('financial_year')[['num_bids']]
    .sum()
    .reset_index()
)

bids_by_fy['n_tenderers'] = (
    awarded.groupby('financial_year')[['successful_tenderer']]
    .nunique()
    .reset_index()['successful_tenderer']
)

bids_by_fy['avg_bids_per_tenderer'] = bids_by_fy['num_bids'] / bids_by_fy['n_tenderers']

full_df = full_df.merge(bids_by_fy[['financial_year', 'n_tenderers']], on='financial_year', how='left')
full_df[['financial_year', 'n_tenderers']].head()

,financial_year,n_tenderers
0,1985-1986,NaN
1,1985-1986,NaN
2,1985-1986,NaN
3,1985-1986,NaN
4,1985-1986,NaN


In [ ]:
import geopandas as gpd

gdf = gpd.read_file("../data/hongkong_districts.gpkg")
gdf = gdf.to_crs("EPSG:4326")

import geopandas as gpd
from shapely.geometry import Point

# Tag each land sale with a district via a spatial join on its lat/long
geo_filtered = full_df.dropna(subset=['latitude', 'longitude']).copy()
geo_filtered['geometry'] = [Point(xy) for xy in zip(geo_filtered['longitude'], geo_filtered['latitude'])]
geo_filtered = gpd.GeoDataFrame(geo_filtered, geometry='geometry', crs='EPSG:4326')

joined = gpd.sjoin(geo_filtered, gdf[['ENAME', 'geometry']], how='left', predicate='within')
joined = joined[~joined.index.duplicated(keep='first')]

full_df['district'] = None
full_df.loc[joined.index, 'district'] = joined['ENAME']
print('Rows without a matched district:', full_df['district'].isna().sum(), '/', full_df.shape[0])

region_map = {
    # Hong Kong Island
    'CENTRAL & WESTERN': 'Hong Kong Island',
    'EASTERN': 'Hong Kong Island',
    'SOUTHERN': 'Hong Kong Island',
    'WAN CHAI': 'Hong Kong Island',

    # Kowloon
    'KOWLOON CITY': 'Kowloon',
    'KWUN TONG': 'Kowloon',
    'SHAM SHUI PO': 'Kowloon',
    'WONG TAI SIN': 'Kowloon',
    'YAU TSIM MONG': 'Kowloon',

    # New Territories
    'ISLANDS': 'New Territories',
    'KWAI TSING': 'New Territories',
    'NORTH': 'New Territories',
    'SAI KUNG': 'New Territories',
    'SHA TIN': 'New Territories',
    'TAI PO': 'New Territories',
    'TSUEN WAN': 'New Territories',
    'TUEN MUN': 'New Territories',
    'YUEN LONG': 'New Territories'
}

full_df['region'] = full_df['district'].map(region_map)

print("District missing:", full_df['district'].isna().sum())
print("Region missing:", full_df['region'].isna().sum())

Rows without a matched district: 151 / 1007
District missing: 151
Region missing: 151
Series([], Name: count, dtype: int64)


In [29]:
## 8) Keep the original use categories and preserve the full sample
# The original exploration step filtered to residential-only rows. This notebook deliberately does not.
# All downstream feature engineering is applied on the full land-sale table with use_category_normalized retained.

feature_cols = [
    'premium_hkd_m', 'num_bids', 'successful_tenderer', 'buyer_normalized', 'buyer_type',
    'use_category_normalized', 'latitude', 'longitude', 'parent_developer', 'award_date',
    'sale_regime', 'site_area_sqm', 'lot_number', 'financial_year', 'location', 'max_gfa',
    'accommodation_value', 'lot_number_normalized', 'in_als', 'als_site_area_sqm',
    'Estimated Earliest Site Available Date', 'Year', 'Deflator', 'region',
    'adj_premium_hkd_m', 'adj_premium_per_sqm', 'FEDFUNDS', 'RGDP(Billions)', 'n_tenderers'
]

final_feature_table = full_df[feature_cols].copy()
print("Final feature table shape:", final_feature_table.shape)
print("Columns:", list(final_feature_table.columns))
final_feature_table.head()

Final feature table shape: (1007, 29)
Columns: ['premium_hkd_m', 'num_bids', 'successful_tenderer', 'buyer_normalized', 'buyer_type', 'use_category_normalized', 'latitude', 'longitude', 'parent_developer', 'award_date', 'sale_regime', 'site_area_sqm', 'lot_number', 'financial_year', 'location', 'max_gfa', 'accommodation_value', 'lot_number_normalized', 'in_als', 'als_site_area_sqm', 'Estimated Earliest Site Available Date', 'Year', 'Deflator', 'region', 'adj_premium_hkd_m', 'adj_premium_per_sqm', 'FEDFUNDS', 'RGDP(Billions)', 'n_tenderers']


,premium_hkd_m,num_bids,successful_tenderer,buyer_normalized,buyer_type,use_category_normalized,latitude,longitude,parent_developer,award_date,...,als_site_area_sqm,Estimated Earliest Site Available Date,Year,Deflator,region,adj_premium_hkd_m,adj_premium_per_sqm,FEDFUNDS,RGDP(Billions),n_tenderers
0,12.10,NaN,NaN,NaN,NaN,industrial,22.382991,114.205675,NaN,1985-04-18,...,NaN,NaN,1985,39.752975,New Territories,0.304380,61.928735,8.27,83.952508,NaN
1,701.28,NaN,NaN,NaN,NaN,commercial_residential_mixed,22.277606,114.165861,NaN,1985-04-18,...,NaN,NaN,1985,39.752975,Hong Kong Island,17.640944,1650.228609,8.27,83.952508,NaN
2,45.50,NaN,NaN,NaN,NaN,residential,22.392917,114.187430,NaN,1985-04-18,...,NaN,NaN,1985,39.752975,New Territories,1.144568,97.742820,8.27,83.952508,NaN
3,27.00,NaN,NaN,NaN,NaN,residential,22.391337,114.188440,NaN,1985-04-18,...,NaN,NaN,1985,39.752975,New Territories,0.679194,71.381445,8.27,83.952508,NaN
4,1.80,NaN,NaN,NaN,NaN,residential,22.394886,114.188673,NaN,1985-04-18,...,NaN,NaN,1985,39.752975,New Territories,0.045280,47.166281,8.27,83.952508,NaN


In [15]:
final_feature_table = final_feature_table[final_feature_table['Year'] >= 1999]
vars = ['adj_premium_per_sqm', 'latitude', 'longitude', 'FEDFUNDS', 'RGDP(Billions)', 'in_als', 'sale_regime', 'max_gfa', 'n_tenderers', 'award_date', 'accommodation_value']
regression = final_feature_table[vars]
regression.dropna().shape

(203, 11)

### Transformation 2: adj_premium_per_sqft_gfa

* Only Residential 
* Land size in sqft as well 

In [ ]:
## Add a new feature adj_premium_per_sqft_gfa
trans1 = final_feature_table.copy()
trans1['adj_premium_per_sqft_gfa'] = (trans1['adj_premium_hkd_m'] * 1000000)/ trans1['max_gfa']

## Transform site area
trans1['site_area_sqft'] = trans1['site_area_sqm'] * 10.7639
trans1.drop(columns={'site_area_sqm'}, inplace=True)


## Filter Use Case Normalised
trans1 = trans1[trans1['use_category_normalized'].str.contains('residential')]
vars = ['adj_premium_per_sqft_gfa', 'latitude', 'longitude', 'FEDFUNDS', 'RGDP(Billions)', 'in_als', 'sale_regime', 'n_tenderers', 'award_date', 'accommodation_value', 'site_area_sqft', 'region']

print(trans1[vars].dropna().shape)
trans1[vars].dropna().head()

# trans1[vars].dropna().to_csv("../data/transformation1.csv", index=False)

(157, 12)


### Transformation 2: Drop `max_gfa` and `accomodation_value`

In [ ]:
trans2 = final_feature_table.copy()

## Filter Use Case Normalised
trans2 = trans1[trans1['use_category_normalized'].str.contains('residential')]

## keeping either of max_gfa and accommodation_value results in 157 observations
vars = ['adj_premium_per_sqm', 'latitude', 'longitude', 'FEDFUNDS', 'RGDP(Billions)', 'in_als', 'sale_regime', 'n_tenderers', 'award_date', 'region']

print(trans2[vars].dropna().shape)
# trans2[vars].dropna().to_csv("../data/transformation2.csv", index=False)
trans2[vars].dropna().head()

(249, 10)


,adj_premium_per_sqm,latitude,longitude,FEDFUNDS,RGDP(Billions),in_als,sale_regime,n_tenderers,award_date,region
498,807.151900,22.275904,114.183839,5.45,162.655118,0,govt-led,4.0,1998-04-17,Hong Kong Island
500,1036.994510,22.504861,114.128737,5.45,162.655118,0,govt-led,4.0,1998-04-22,New Territories
503,2291.354912,22.283605,114.224674,5.49,162.655118,0,govt-led,4.0,1998-05-22,Hong Kong Island
508,452.522264,22.280948,114.228787,5.54,162.655118,0,govt-led,4.0,1998-07-15,Hong Kong Island
510,195.610425,22.306608,114.262269,5.07,162.655118,0,govt-led,4.0,1998-10-07,New Territories
